In [2]:
import pandas as pd

# 파일 읽기
df = pd.read_csv("DCT_image_1.txt", header=None)

# 0과 1만 포함된 행 검사
invalid = df[~df[0].astype(str).str.fullmatch(r'[01]+')]

# 결과 출력
if invalid.empty:
    print("모든 데이터가 0과 1로만 구성됨")
else:
    print("잘못된 데이터:")
    print(invalid)

잘못된 데이터:
                                                       0
1024   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
1025   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
1026   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
1027   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
1028   xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
...                                                  ...
16379  xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
16380  xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
16381  xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
16382  xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
16383  xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...

[15360 rows x 1 columns]


In [14]:
import os

matlab_file = "./bb/DCT_image_3.txt"

if os.path.exists(matlab_file):
    print(f"✅ 파일을 찾았습니다: {os.path.abspath(matlab_file)}")
else:
    print("❌ 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    
    # 디버깅을 위해 현재 폴더 구조 탐색
    print(f"\n[참고] 현재 실행 경로: {os.getcwd()}")
    parent_dir = "."
    if os.path.exists(parent_dir):
        print(f"'{parent_dir}' 폴더 안의 파일 목록:", os.listdir(parent_dir))

✅ 파일을 찾았습니다: c:\Users\smoh\Desktop\26-1\Verilog\proj2\cur\aa\bb\DCT_image_3.txt


In [ ]:
import numpy as np

matlab_file  = "./DCT_image_3.txt"        # generate 골든 (12bit/줄)
verilog_file = "./bb/DCT_image_3.txt"   # 베릴로그 출력 (12bit/줄, 새 TB)
output_report_file = "debugging_report_fixed.txt"

# 12bit 2의보수 1줄 -> 정수
def parse_bin12(b):
    b = b.strip().replace(" ", "")
    if len(b) != 12 or any(c not in "01" for c in b):
        return None
    v = int(b, 2)
    return v - 4096 if v & 0x800 else v

# 두 파일 모두 '12bit = 1계수 = 1줄' 로 로드 (262144개)
def load_12bit(path):
    out = []
    with open(path) as f:
        for line in f:
            v = parse_bin12(line)
            if v is not None:
                out.append(v)
    return out

mat_list = load_12bit(matlab_file)
v_list   = load_12bit(verilog_file)
print(f"MATLAB 계수 수: {len(mat_list)}, 베릴로그 계수 수: {len(v_list)} (기대 262144)")

# 계수 스트림 순서 = word0[ch0..ch15], word1[...], ...
# word w -> 블록 b=w//16, 행 r=w%16. 블록순서 l(32)->k(32). 계수 c=열.
def to_image(flat):
    img = np.zeros((512,512), dtype=np.int32)
    idx = 0
    for l in range(32):
        for k in range(32):
            for r in range(16):          # word 내 행
                for c in range(16):      # 열 계수
                    if idx < len(flat):
                        img[16*l+r, 16*k+c] = flat[idx]; idx += 1
    return img

mat = to_image(mat_list)
ver = to_image(v_list)
diff = ver - mat
err = int(np.count_nonzero(diff))

with open(output_report_file, "w", encoding="utf-8") as rf:
    def w(t): print(t); rf.write(t+"\n")
    w("="*60)
    w("   RTL vs MATLAB 골든 DCT 오차 분석 (12bit/줄 기준)")
    w("="*60)
    w(f"  전체 불일치 계수 개수: {err} 개")
    if err == 0:
        w("\n  [성공] 베릴로그 RTL = 매트랩 골든 100% 일치")
    else:
        br=bc=None
        for i in range(32):
            for j in range(32):
                if np.any(diff[i*16:(i+1)*16, j*16:(j+1)*16] != 0):
                    br,bc=i,j; break
            if br is not None: break
        w(f"  최초 오차 블록: ({br},{bc})")
        rs,cs = br*16, bc*16
        np.set_printoptions(linewidth=200, formatter={'int':'{:5d}'.format})
        w("\n[1] MATLAB 골든\n"+np.array2string(mat[rs:rs+16, cs:cs+16]))
        w("\n[2] 베릴로그 RTL\n"+np.array2string(ver[rs:rs+16, cs:cs+16]))
        w("\n[3] 오차(베릴로그-매트랩)\n"+np.array2string(diff[rs:rs+16, cs:cs+16]))

MATLAB 계수 수: 262144, 베릴로그 계수 수: 262144 (기대 262144)
   RTL vs MATLAB 골든 DCT 오차 분석 (12bit/줄 기준)
  전체 불일치 계수 개수: 104251 개
  최초 오차 블록: (0,0)

[1] MATLAB 골든
[[  160  -184   -16    -3   -12    -5    -4   -10    -2    -2    -5    -5    -3    -4    -3    -3]
 [  106  -157    48     7    -1    -9     2    -6    -1    -1    -1     0    -1    -3     0     3]
 [   35   -25    -5     6    12    -9     8    -4    -1    -1    -1     0     1    -2     0     0]
 [   28   -18   -10     8    -2    -1     5    -1    -1     0     0    -1     2    -1     1    -1]
 [    0   -19     5    -9    -1     1    -1    -3    -2    -2     0     0    -1     2     0    -2]
 [    5     0    -8    -1    -1    -4    -3     0    -1    -2    -1     1    -2    -1     0     0]
 [    0     0     0     2     0     0     0     0     0    -1    -2    -1    -1    -1     0    -1]
 [   -2     1     1     2    -1     2     0    -1     0    -1     0    -1    -1     0    -3    -1]
 [   -1     0    -1     1    -1    -1     0     0     0 